# Base Graph Creation from PostGIS

This notebook creates a maritime navigation graph from S-57 data stored in a PostGIS database. It defines an area of interest between two ports, filters relevant ENCs, generates a navigable grid, and constructs a NetworkX graph for pathfinding.

#### Workflow Overview

1. **Define Area of Interest** - Select two ports and create expanded bounding box
2. **Filter ENCs** - Query PostGIS for charts intersecting the area of interest
3. **Generate Navigable Grid** - Combine S-57 layers into single polygon
4. **Construct Graph** - Build NetworkX graph with configurable node spacing
5. **Calculate Route** - Compute shortest path using A* pathfinding
6. **Persist Results** - Save graph and route to both GeoPackage and PostGIS

#### Data Flow

```
PostGIS → ENC Filtering → Navigable Grid → NetworkX Graph → Route → GPKG/PG Outputs
```

#### Expected Outputs

- **Navigation Graph**: Nodes and edges saved to both formats (count varies by spacing, see docs/reference/technical-specs.md)
- **Base Route**: Shortest path geometry between ports
- **Benchmarks**: Timing metrics appended to CSV
- **Visualizations**: Interactive maps showing ENCs, grid, and route

#### Required Data

This notebook requires:
1. **ENC Data**: S-57 charts converted to PostGIS format
2. **Database Schema**: Schema containing S-57 layers (e.g., `enc_west`)
3. **Port Data**: Standard port definitions (included with package)
4. **Connection**: PostgreSQL credentials configured in `.env` file

**Setup Instructions:** See `docs/getting-started/setup.md`
**Troubleshooting:** See `docs/reference/troubleshooting.md`

## 1. Configuration

Centralized parameter configuration for the graph creation workflow. All user-adjustable settings are defined in the code cell below.

In [ ]:
# =============================================================================
# NOTEBOOK CONFIGURATION - Adjust these parameters before running
# =============================================================================

# --- Port Selection ---
departure_port_name = "Los Angeles"
arrival_port_name = "San Francisco"
aoi_expansion_nm = 24  # Buffer around ports in nautical miles

# --- Data Source (PostGIS) ---
data_schema = "enc_west"  # Schema name containing S-57 data

# --- Graph Parameters ---
primary_layer = "seaare"  # Primary navigable layer (e.g., seaare, fairwy)
reduce_distance_nm = 3  # Safety buffer to shrink navigable area (0 = no reduction, 3 = 3nm buffer)
spacing_nm = 0.3  # Node spacing in nautical miles (affects node count, see APPENDIX and docs/reference/technical-specs.md)
keep_largest_component = True  # Remove isolated node clusters that would cause routing errors
bridge_components = True # connects nearby disconnected graph components within max_edge_factor * spacing_nm distance to fix gaps caused by numerical precision issues in finer grids

# --- Output Settings ---
save_to_gpkg = True  # Save graph to GeoPackage file for portability
save_to_postgis = True  # Save graph to PostGIS database for server-side queries
output_graph_gpkg_name = "base_graph_pg.gpkg"  # GeoPackage filename
output_graph_pg_prefix = "base_graph_la_sf"  # PostGIS table prefix (creates ..._nodes and ..._edges tables)
drop_existing_pg_tables = True  # Drop existing PostGIS tables before saving (careful: destructive!)

# --- Route Settings ---
calculate_route = True  # Calculate optimal maritime route between ports
route_name = "base_route"  # Name for saved route
route_schema = "routes"  # Schema for route storage
route_table = "base_routes"  # Table name for routes
save_route_overwrite = True  # Overwrite existing route with same name

# --- Configuration Summary ---
print("=" * 70)
print("✓ Configuration loaded successfully!")
print("=" * 70)
print(f"📍 Route: {departure_port_name} → {arrival_port_name}")
print(f"📊 Data schema: {data_schema}")
print(f"🛣️  Graph spacing: {spacing_nm} NM (node resolution)")
print(f"📐 Safety buffer: {reduce_distance_nm} NM")
print(f"💾 Saving to:")
if save_to_gpkg:
    print(f"   - GeoPackage: {output_graph_gpkg_name}")
if save_to_postgis:
    print(f"   - PostGIS: tables '{output_graph_pg_prefix}_nodes' & '{output_graph_pg_prefix}_edges'")
if calculate_route:
    print(f"🗺️  Route output: {route_table}.{route_name}")
print("=" * 70)

### 1.1 Parameter Quick Reference

This notebook uses several parameters to control graph creation and routing quality.

**spacing_nm** - Node density (0.02-0.5 NM, default: 0.3)
- Controls navigation node precision. Lower = more detailed but slower processing.

**reduce_distance_nm** - Safety buffer from hazards (0-5 NM, default: 3).
 Shrinks navigable area to maintain safe clearance from hazards and shore.
- **3 NM**: Standard maritime safety (default, recommended)
- **0 NM**: No buffer (trust chart edges)
- **5 NM**: Conservative margin (uncertain data quality)

**primary_layer** - Navigable area source (default: seaare)
- Options: seaare (sea areas), fairwy (shipping lanes), drgare (dredged channels)

**keep_largest_component** - Filter disconnected clusters (default: True)
- Removes isolated node groups to prevent routing failures.

**aoi_expansion_nm** - Port buffer size (5-50 NM, default: 24)
- Determines how much water is included around ports for route calculations.

**For detailed parameter explanations and trade-offs, see APPENDIX below.**
**For backend selection guidance, see `docs/getting-started/setup.md` and `docs/user-guides/database-backend-guide.md`.**

### 1.2 Imports & Environment Validation

Validate database connectivity and required S-57 layers before proceeding with graph creation.

In [ ]:
import sys
import os
import time
from pathlib import Path
from dotenv import load_dotenv
import plotly.io as pio
import pandas as pd
import plotly.express as px

# --- Fix PROJ_LIB Path (Common Conda/Jupyter Issue) ---
# Ensure GDAL/PROJ can find the coordinate database
conda_prefix = sys.prefix
possible_proj_lib = os.path.join(conda_prefix, 'share', 'proj')
if os.path.exists(possible_proj_lib):
    os.environ['PROJ_LIB'] = possible_proj_lib

# --- Setup Python Environment ---
# Get project root for .env file loading
project_root = Path.cwd().parent.parent

# --- Validate Project Root ---
if not (project_root / "src" / "nautical_graph_toolkit").exists():
    raise FileNotFoundError(
        f"❌ Invalid project root: {project_root}\n"
        f"   Expected to find src/nautical_graph_toolkit/\n"
        f"   Notebooks must be run from docs/notebooks/ directory"
    )

# --- Load and Validate Environment Variables ---
env_file = project_root / ".env"
if not env_file.exists():
    print(f"⚠️  .env file not found at: {env_file}")
    print(f"   Copy .env.example to .env and configure:")
    print(f"   cp {project_root}/.env.example {env_file}")
    raise FileNotFoundError("Required .env file missing")

load_dotenv(env_file)

pio.renderers.default = "notebook_connected"

# Import maritime module components
from nautical_graph_toolkit.core.s57_data import ENCDataFactory, S57AdvancedConfig
from nautical_graph_toolkit.utils.port_utils import Boundaries, PortData
from nautical_graph_toolkit.utils.plot_utils import PlotlyChart
from nautical_graph_toolkit.utils.notebook_utils import BenchmarkLogger, load_estimates
from nautical_graph_toolkit.core.graph import BaseGraph
from nautical_graph_toolkit.core.pathfinding_lite import Route

# --- Define Output Directory ---
# Create output directory for saving results
output_dir = Path.cwd() / 'output'
output_dir.mkdir(exist_ok=True)

# --- Define Data Source (PostGIS Database Connection) ---
# PostGIS uses dict of connection params, file-based backends use Path
# These credentials are loaded from the .env file
db_params = {
    'dbname': os.getenv('DB_NAME'),
    'user': os.getenv('DB_USER'),
    'password': os.getenv('DB_PASSWORD'),
    'host': os.getenv('DB_HOST'),
    'port': os.getenv('DB_PORT')
}

print(f"Output directory: {output_dir}")
print(f"Data source: PostGIS database '{db_params['dbname']}'")

# --- Performance Tracking ---
logger = BenchmarkLogger()
logger.configure_base_graph(graph_mode='base',
                            spacing_nm=spacing_nm,
                            reduce_distance_nm=reduce_distance_nm,
                            aoi=f'{departure_port_name} - {arrival_port_name}',
                            db_schema=data_schema)

logger.set_result('workflow', 'graph_PostGIS_v2')
logger.set_result('data_source', 'PostGIS')
logger.set_result('bridge_components', bridge_components)

# --- Load Historical Time Estimates ---
estimate = load_estimates(
    notebook='graph_PostGIS_v2',
    graph_mode='base',
    spacing_nm=spacing_nm,
    backend='PostGIS'
)
if estimate:
    print(f"⏱️  Estimated duration: {estimate['mean']:.1f} ± {estimate['std_dev']:.1f}s")
    print(f"   Based on {estimate['count']} previous runs")
    print(f"   Range: {estimate['min']:.1f}s - {estimate['max']:.1f}s")
else:
    print("⏱️  No historical estimates available (first run)")

# --- DATABASE CONNECTION VALIDATION ---
try:
    import psycopg2
    test_conn = psycopg2.connect(**db_params)
    test_conn.close()
    print("✅ PostGIS connection successful")
except Exception as e:
    print(f"❌ Database connection failed: {e}")
    print("Check .env file configuration:")
    print(f"  - DB_NAME: {db_params.get('dbname')}")
    print(f"  - DB_HOST: {db_params.get('host')}")
    print(f"  - DB_PORT: {db_params.get('port')}")
    print(f"  - DB_USER: {db_params.get('user')}")
    raise

print("✓ All imports loaded successfully!")

### 1.3 Workflow Context

**Pipeline Position**: Step 2 of 3
1. **Data Import** (`import_s57.ipynb`) - Convert S-57 ENCs to GeoPackage, PostGIS, or SpatiaLite backend
2. **Graph Construction** (This notebook) - Build maritime navigation graph from backend data
3. **Weighting & Routing** (`graph_weighted_directed_Postgis_v2.ipynb`) - Add edge weights and optimize routes

**Prerequisites**:
- Completed `import_s57.ipynb` with PostGIS backend
- PostgreSQL database with schema `us_enc_all` (or your schema name) containing converted S-57 layers
- Connection parameters configured in `.env` file

**Outputs**:
- Navigation graph (nodes + edges) saved to both GeoPackage and PostGIS formats
- Base route between departure and arrival ports
- Performance benchmarks appended to `benchmark_graph_base.csv`

**Next Steps**:
- Run `graph_weighted_directed_Postgis_v2.ipynb` to add edge weights based on distance, depth, and weather
- Or switch backends by running `graph_GeoPackage_v2.ipynb` or `graph_SpatiaLite_v2.ipynb` instead
- Use the graph for pathfinding, analysis, and server-side PostGIS queries

### 1.4 Initialize ENC Data Factory

Confirm required S-57 layers exist in the specified schema.

In [ ]:
# =============================================================================
# SCHEMA AND LAYER VALIDATION
# =============================================================================

required_layers = ['seaare', 'lndare', 'fairwy', 'drgare', 'tsslpt', 'prcare']

try:
    pg_factory = ENCDataFactory(source=db_params, schema=data_schema)
    manager = pg_factory.manager
    
    # Get summary of available layers
    layers_df = manager.get_layers_summary(include_empty=False)
    available_layers = layers_df['Acronym'].tolist()
    
    missing_layers = [layer for layer in required_layers if layer not in available_layers]
    
    if missing_layers:
        print(f"❌ Required layers missing: {', '.join(missing_layers)}")
        print(f"   Check schema '{data_schema}' in database '{db_params['dbname']}'")
        print(f"   Expected layers: {', '.join(required_layers)}")
        print(f"   Available layers: {', '.join(available_layers)}")
        print(f"\n🔧 To fix:")
        print(f"   1. Convert S-57 data using import_s57.ipynb")
        print(f"   2. Specify correct schema in notebook configuration")
        raise ValueError(f"Layers {missing_layers} not found in schema '{data_schema}'")
    
    print(f"✅ All {len(required_layers)} required layers present in schema '{data_schema}'")

except Exception as e:
    print(f"❌ Schema validation failed: {e}")
    raise

## 2. Define Area of Interest (AOI)

This first step defines the geographic scope for our graph. We select two ports and create an expanded bounding box around them to ensure all relevant navigational data is included.

In [ ]:
# --- Define Area of Interest by Selecting Two Ports ---
# Get port data and create a bounding box between departure and arrival ports
# The expansion parameter adds a buffer around the ports to include surrounding navigable areas
logger.start_timer('port_selection_boundary')

port  = PortData()
bbox = Boundaries()
port1 = port.get_port_by_name(departure_port_name)
port2 = port.get_port_by_name(arrival_port_name)

# --- Validate Port Selection ---
# Ensure both ports were found in the database before proceeding
if port1.empty or port2.empty:
    raise ValueError(f"Could not find one or both ports. Please check the names.")
else:
    print(port.format_port_string(port1))
    print(port.format_port_string(port2))
    # Create expanded boundary around the two ports
    # date_line=True handles cases where routes cross the International Date Line
    port_bbox = bbox.create_geo_boundary(geometries = [port1.geometry, port2.geometry],
                                         expansion=aoi_expansion_nm,
                                         date_line=True)

elapsed = logger.end_step('port_selection_boundary')
print(f"Port selection and boundary creation took: {elapsed:.2f}s")
port_bbox

### 2.1 Visualize the Area of Interest

Here, we plot the selected ports and the calculated boundary on a map to visually confirm our area of interest.

In [ ]:
# --- Visualize Ports on Interactive Map ---
# Create a Plotly map and add both ports as markers
# This helps verify port locations before proceeding with graph creation
ply = PlotlyChart()
ply_fig = ply.create_base_map(mapbox_token=os.getenv('MAPBOX_TOKEN'))
ply.plotly_base_config(ply_fig)
port1_df = port.get_port_details_df(port1)
port2_df = port.get_port_details_df(port2)
# Add departure port (Los Angeles) in blue
ply.add_single_port_trace(ply_fig, port1, name=port1['PORT_NAME'], color='blue')
# Add arrival port (San Francisco) in red
ply.add_single_port_trace(ply_fig, port2, name=port2['PORT_NAME'], color='red')
ply_fig.show()

In [ ]:
# --- Add Boundary to Map Visualization ---
# Display the expanded boundary box on the map to show our area of interest
ply.add_boundary_trace(ply_fig, port_bbox)
ply_fig.show()

## 3. ENC Data Preparation

With the AOI defined, we now query the PostGIS database to find all Electronic Navigational Charts (ENCs) that intersect with our boundary. This ensures we only process relevant chart data, which is critical for performance.

In [ ]:
# --- Initialize ENC Data Factory for PostGIS Backend ---
# The factory provides a unified interface for accessing ENC data
# regardless of backend (PostGIS/GeoPackage/SpatiaLite).
logger.start_timer('enc_filtering')

pg_factory = ENCDataFactory(source=db_params, schema=data_schema)

# --- Filter ENCs by Boundary ---
# Step 1: Get the list of ENC names that intersect with our area of interest
# This ensures we only process relevant chart data, critical for performance
enc_names_in_boundary = pg_factory.get_encs_by_boundary(port_bbox.geometry.iloc[0])

# Step 2: Get the bounding box GeoDataFrame for only those filtered ENCs
# This provides geographic extents for visualization
enc_bbox_gdf = pg_factory.get_enc_bounding_boxes(enc_names_in_boundary)

# --- Visualize ENC Coverage on Map ---
# Step 3: Add the ENC boundaries to our map to verify coverage
# Different usage bands (1-6) represent different chart scales/detail levels
ply.add_enc_bbox_trace(figure=ply_fig, bbox_df=enc_bbox_gdf, usage_bands=[1,2,3,4,5,6])

elapsed = logger.end_step('enc_filtering')
print(f"ENC filtering took: {elapsed:.2f}s")
ply_fig.show()

## 4. Graph Generation and Pathfinding

### 4.1 Create Navigable Grid

This step queries the S-57 layer and other navigable layers to create a single polygon representing all navigable water within the AOI. The grid combines primary sea areas (seaare) with supplementary navigable layers (fairways, channels) while excluding obstacles (land, constructions).

In [ ]:
# --- Initialize BaseGraph for PostGIS Backend ---
# BaseGraph provides the core graph creation functionality and works
# with any data backend (PostGIS, GeoPackage, SpatiaLite).
# It handles:
#   - Querying S-57 layers from the data source
#   - Creating navigable grids from chart data
#   - Building NetworkX graphs with proper connectivity
#   - Saving graphs to various formats

logger.start_timer('grid_creation')

pg_bg = BaseGraph(data_factory=ENCDataFactory(db_params, schema=data_schema),
                  graph_schema_name="graph")

# --- Create Navigable Grid ---
# This step queries the S-57 layer and other navigable layers to create 
# a single polygon representing all navigable water within the AOI.
#
# The create_base_grid method:
# 1. Queries primary navigable layer for navigable areas
# 2. Adds supplementary navigable layers (fairways, channels, traffic lanes)
# 3. Subtracts obstacle layers (land, constructions)
# 4. Optionally reduces the navigable area by reduce_distance_nm to maintain
#    safe distance from hazards
#
# Parameters:
#   - port_boundary: Geographic boundary defining the area of interest
#   - departure_port/arrival_port: Used to ensure connectivity near ports
#   - layer_table: Primary navigable layer (typically "seaare")
#   - reduce_distance_nm: Safety buffer to shrink navigable area (0 = no reduction)
grid = pg_bg.create_base_grid(port_boundary=port_bbox,
                              departure_port=port1,
                              arrival_port=port2,
                              layer_table=primary_layer,
                              reduce_distance_nm=reduce_distance_nm)

elapsed = logger.end_step('grid_creation')
print(f"Grid creation took: {elapsed:.2f}s")

### 4.2 Visualize Grid Components

We plot the different components of the generated grid to understand coverage and verify that the navigable areas are correctly identified. The visualization shows the main grid (sea areas), extra grids (fairways and channels), and the final combined grid used for graph creation.

In [ ]:
# --- Visualize Grid Components ---
# We plot the different components of the generated grid to understand coverage:
# - main_grid (red): Primary sea area polygons from 'seaare' layer
# - extra_grids (green): Additional navigable areas (fairways, channels, etc.)
# - combined_grid (blue): Final merged navigable polygon used for graph creation
ply_grid = ply.create_base_map(mapbox_token=os.getenv('MAPBOX_TOKEN'))
ply.plotly_base_config(ply_grid)
ply.add_grid_trace(ply_grid, grid_geojson=grid["main_grid"], color="red")
ply.add_grid_trace(ply_grid, grid_geojson=grid["extra_grids"], color="green")
ply.add_grid_trace(ply_grid, grid_geojson=grid["combined_grid"], color="blue")
ply_grid.show()

### 4.3 Construct Graph from Grid

This is the core graph creation step. It populates the navigable grid polygon with a dense network of nodes and edges that form the basis for pathfinding. The graph is created using configurable node spacing (in nautical miles) which determines the precision of the routing resolution.

In [ ]:
# --- Construct Graph from Grid ---
# This is the core graph creation step. It populates the navigable grid polygon 
# with a dense network of nodes and edges.
#
# The create_base_graph method:
# 1. Generates a regular grid of nodes within the navigable polygon
#    - Node spacing: configurable in NM (finer spacing = more nodes = better precision)
# 2. Creates edges between adjacent nodes (8-connectivity: N, S, E, W, NE, NW, SE, SW)
# 3. Calculates edge lengths in nautical miles for distance-based routing
# 4. Optionally filters to keep only the largest connected component
#    - Removes isolated node clusters that would cause pathfinding failures
#    - Ensures start and end nodes are always in the same connected network
#
# Performance considerations:
#   - 0.3 NM spacing: fast, suitable for ocean routing
#   - 0.15 NM spacing: slower, better for detailed coastal/harbor routing
#   - keep_largest_component=True: Recommended to prevent routing errors

logger.start_timer('graph_creation')
G = pg_bg.create_base_graph(grid["combined_grid"], 
                            spacing_nm=spacing_nm,
                            keep_largest_component=keep_largest_component,
                            bridge_components=bridge_components)

elapsed = logger.end_step('graph_creation')
logger.set_result('node_count', G.number_of_nodes())
logger.set_result('edge_count', G.number_of_edges())
print(f"Graph has {G.number_of_nodes():,} nodes and {G.number_of_edges():,} edges")
print(f"Graph creation took: {elapsed:.2f}s")

### 4.4 Save Graph to GeoPackage

For portability and interoperability, we export the graph to GeoPackage format. This single-file format is ideal for sharing, archiving, and opening in standard GIS software like QGIS and ArcGIS without requiring a database server.

In [ ]:
# --- Save Graph to GeoPackage File ---
# For portability and interoperability, save the graph to GeoPackage format.
#
# GeoPackage Advantages:
# ----------------------
# 1. Single-file format: Easy to share, archive, and version control
# 2. No server required: Works offline without database setup
# 3. Cross-platform: Opens in QGIS, ArcGIS, and other GIS software
# 4. Open standard: OGC-compliant format with wide tool support
#
# The saved GeoPackage contains two layers:
#   - nodes: Point geometries with attributes (node_id, lon, lat)
#   - edges: LineString geometries with attributes (source, target, length)
#
# Use cases:
#   - Visualization in QGIS for quality assurance
#   - Sharing graphs with collaborators (no database access needed)
#   - Archiving graph snapshots for reproducibility
#   - Loading into other analysis tools (R, QGIS processing, etc.)

if save_to_gpkg:
    logger.start_timer('save_gpkg')

    output_file = output_dir / output_graph_gpkg_name
    pg_bg.save_graph_to_gpkg(G, output_file)

    elapsed = logger.end_step('save_gpkg')
    print(f"Saving to GeoPackage took: {elapsed:.2f}s")
    print(f"Saved to: {output_file}")
else:
    print("Saving to GeoPackage skipped (save_to_gpkg = False)")

### 4.5 Save Graph to PostGIS

We store the graph in PostGIS for server-side spatial analysis and multi-user access. This enables integration with other workflows, centralized storage, and the ability to run server-side spatial queries on graph data.

In [ ]:
# --- Save Graph to PostGIS Database ---
# Store the graph in PostGIS for server-side spatial analysis and integration
# with other database workflows.
#
# PostGIS Advantages:
# -------------------
# 1. Server-side queries: Leverage PostGIS spatial functions on graph data
# 2. Multi-user access: Share graphs across team via database server
# 3. Integration: Combine graph with other spatial data in SQL queries
# 4. Persistence: Centralized storage with backup/recovery capabilities
#
# The method creates two tables in the specified schema:
#   - {prefix}_nodes: Point geometries with spatial index
#   - {prefix}_edges: LineString geometries with spatial index
#
# WARNING: drop_existing=True will delete any existing tables with the same name!
#
# Use cases:
#   - Server-side routing queries using PostGIS functions
#   - Integration with web services and APIs
#   - Combining graph data with live AIS ship tracking
#   - Multi-user collaborative analysis

if save_to_postgis:
    logger.start_timer('save_postgis')
    pg_bg.save_graph_to_postgis(graph=G,
                                table_prefix=output_graph_pg_prefix,
                                drop_existing=drop_existing_pg_tables)
    elapsed = logger.end_step('save_postgis')
    print(f"Saving to PostGIS took: {elapsed:.2f}s")
    print(f"Saved tables: {output_graph_pg_prefix}_nodes, {output_graph_pg_prefix}_edges")
else:
    print("Saving to PostGIS skipped (save_to_postgis = False)")

### 4.6 Perform Base Routing

With the graph created, we compute the optimal maritime route between the two ports using the A* pathfinding algorithm. This algorithm finds the shortest path efficiently by combining actual path cost with a heuristic estimate of distance to the goal.

In [ ]:
# --- Perform Base Routing (Shortest Path Calculation) ---
# With the graph created, compute the optimal maritime route between ports
# using the A* pathfinding algorithm.
#
# A* Algorithm:
# -------------
# A* is an informed search algorithm that finds the shortest path efficiently by:
# 1. Using actual path cost (distance traveled so far)
# 2. Adding heuristic estimate (straight-line distance to goal)
# 3. Always exploring most promising nodes first
# 4. Guaranteeing optimal solution (with admissible heuristic)
#
# The Route class handles:
# ------------------------
# 1. Coordinate mapping: Maps port lat/lon to nearest graph nodes using spatial index
# 2. Validation: Ensures start and end nodes exist and are in same connected component
# 3. Pathfinding: Runs A* with Euclidean distance heuristic
# 4. Geometry creation: Converts node sequence to LineString route geometry
# 5. Distance calculation: Computes total route distance in nautical miles
#
# Current implementation uses base distance weighting (edges weighted by length only).
# Future enhancements can incorporate:
#   - Weather routing (wind, currents, waves)
#   - Traffic separation scheme compliance
#   - Depth restrictions (avoid shallow areas)
#   - Vessel-specific constraints (draft, turning radius)
if calculate_route:
    logger.start_timer('pathfinding')

    route = Route(graph=G, data_manager=pg_factory.manager)
    route_geometry, distance = route.base_route(
        departure_point=port1.geometry,
        arrival_point=port2.geometry
    )
    elapsed = logger.end_step('pathfinding')
    logger.set_result('route_distance_nm', distance)
    print(f"Pathfinding took: {elapsed:.2f}s")
    print(f"Route distance: {distance:.2f} nautical miles")
else:
    print("Route calculation skipped (calculate_route = False)")

### 4.7 Visualize Route

We display the calculated route as a line on the map along with both ports. This provides visual verification that the routing algorithm produced a sensible maritime path between the two locations.

In [ ]:
# --- Visualize Computed Route on Map ---
# Display the calculated route as a line on the map along with both ports.
# This provides a visual verification that the routing algorithm produced
# a sensible maritime path between the two locations.
if calculate_route:
    ply_route = ply.create_base_map(mapbox_token=os.getenv('MAPBOX_TOKEN'))
    ply.plotly_base_config(ply_route)
    # Add the route line
    ply.add_route_trace(figure=ply_route,
                        line=route_geometry,
                        name="Base Route")
    # Add departure port marker
    ply.add_single_port_trace(ply_route, port1, name=port1['PORT_NAME'], color='blue')
    # Add arrival port marker
    ply.add_single_port_trace(ply_route, port2, name=port2['PORT_NAME'], color='red')
    ply_route.show()
else:
    print("Route visualization skipped (calculate_route = False)")

### 4.8 Save Route to Database

Finally, we persist the computed route to PostGIS for future reference and analysis. This enables route comparison, integration with other workflows, and centralized storage of routing results.

In [ ]:
# --- Save Route to PostGIS Database ---
# Store the computed route geometry for future reference and analysis.
#
# Route Storage Benefits:
# -----------------------
# 1. Persistence: Routes remain available across sessions
# 2. Comparison: Compare different routing strategies or parameters
# 3. Analysis: Analyze route characteristics (distance, waypoints, segments)
# 4. Visualization: Load routes in GIS tools for presentation
# 5. Integration: Use routes in other workflows (fuel estimation, ETA calculation)
#
# Routes are saved in a dedicated schema with metadata:
#   - route_name: Identifier for retrieval
#   - geometry: LineString route path
#   - distance: Total distance in nautical miles
#   - timestamp: When route was computed
#
# The overwrite parameter controls behavior when a route with the same name exists:
#   - True: Replace existing route with new computation
#   - False: Raise error if route name already exists

if calculate_route:
    pg_factory.save_route(route_geom=route_geometry,
                          route_name=route_name,
                          schema_name=route_schema,
                          table_name=route_table,
                          overwrite=save_route_overwrite)
    print("Route saved successfully to PostGIS")
    print(f"Access via: SELECT * FROM {route_schema}.{route_table} WHERE route_name = '{route_name}'")
else:
    print("Route saving skipped (calculate_route = False)")

## Performance Summary

This section visualizes the time taken for each step of the pipeline.

In [ ]:
# =============================================================================
# PERFORMANCE SUMMARY
# =============================================================================

# --- Export Benchmark to CSV ---
csv_path = logger.export_benchmark()
print(f"\n💾 Benchmark saved to: {csv_path}")

# --- Load Historical Estimates for Comparison ---
historical = load_estimates(
    notebook='graph_PostGIS_v2',
    graph_mode='base',
    spacing_nm=0.3,
    backend='PostGIS'
)
print("\n")
print(logger.get_current_benchmark_summary())
if historical:
    print(f"\n📊 Historical Performance (Records={historical['count']}):")
    print(f"   Mean: {historical['mean']:.1f}s ± {historical['std_dev']:.1f}s")
    print(f"   Min: {historical['min']:.1f}s, Max: {historical['max']:.1f}s")
print("\n")

# --- Visualize Pipeline Performance ---
fig = logger.visualize_performance(
    title='Base Graph Creation Pipeline Performance (PostGIS)',
    sort_by='time_descending',
    show=True
)


In [ ]:
df = pg_factory.manager.get_table_sizes(include_empty=False,
                                        table_name=None,
                                        schema_name='graph')
df

## APPENDIX: Detailed Documentation

This appendix contains detailed reference information specific to this notebook. For general information about backends, performance benchmarks, and S-57 formats, refer to the external documentation listed in the Cross-References section.

### A.1 Detailed Parameter Explanations

**spacing_nm** - Node density (0.02-0.5 NM)
- Controls how closely spaced nodes are in the graph
- Smaller = more precise routes but slower processing
- Larger = faster but less precise routing
- For BaseGraph: 0.3 NM is recommended for most tasks, 0.5-1.0 NM for ocean crossing
- For detailed work: Use FineGraph notebooks instead (see `graph_fine_*.ipynb`)

See `docs/reference/technical-specs.md` for actual node counts and execution times.

**reduce_distance_nm** - Safety buffer from hazards (0-10 NM)
- Shrinks navigable areas to maintain safe distance from hazards
- PostGIS uses 0 NM by default (no buffer)
- GeoPackage/SpatiaLite typically use 3 NM (standard maritime safety)
- Higher values = longer, safer routes; lower values = shorter routes closer to hazards
- Use 3-5 NM for uncertain data, 0-1 NM for high-confidence charts

**primary_layer** - Navigable area source
- `seaare` = Sea areas (most reliable, recommended)
- `fairwy` = Shipping lanes (traffic separation schemes)
- `drgare` = Dredged channels (maintained waterways)
- Selection affects which water features are considered navigable

**keep_largest_component** - Filter disconnected clusters (True/False)
- True = Removes isolated node groups (prevents routing failures)
- False = Keeps all nodes (maximum coverage but may have failures)
- Recommended: True to ensure start/end ports are always connected

**aoi_expansion_nm** - Port buffer size (5-50 NM)
- Determines how much water is included around ports
- 24 NM = standard (balanced), 10 NM = minimal (faster), 50 NM = maximum
- Larger values include more chart data but slower processing
- Should account for coastal routing, detours around islands/peninsulas

### A.2 Recommended Configurations

**Ocean Routing (Default):**
- spacing_nm = 0.3
- reduce_distance_nm = 0
- primary_layer = seaare
- aoi_expansion_nm = 24

**Harbor Navigation:**
- spacing_nm = 0.1 (use FineGraph instead)
- reduce_distance_nm = 0.5
- primary_layer = seaare
- aoi_expansion_nm = 10

**High Safety/Uncertainty:**
- spacing_nm = 0.3
- reduce_distance_nm = 5
- primary_layer = seaare
- aoi_expansion_nm = 50

### A.3 Performance Notes

Performance varies significantly based on:
- **Node spacing**: Smaller spacing = more nodes = longer processing
- **Hardware**: CPU cores, RAM, disk I/O speed
- **Backend**: PostGIS vs GeoPackage vs SpatiaLite

**For actual benchmark data including:**
- Node and edge counts by spacing
- Execution times for each workflow step
- Backend performance comparison
- Hardware specifications

**See:** `docs/reference/technical-specs.md`

**General Expectations**:
- Grid creation: Scales linearly with area size
- Graph creation: Scales quadratically with node count
- Save operations: Depends on backend and node count
- Pathfinding: Sub-linear scaling with graph size (A* with spatial indexing)

### A.4 Cross-References

- **Backend Guide**: `docs/user-guides/database-backend-guide.md` - Backend selection and tradeoffs
- **Setup Instructions**: `docs/getting-started/setup.md` - Complete environment setup
- **Technical Specs**: `docs/reference/technical-specs.md` - Performance benchmarks and hardware requirements